    <a href="https://colab.research.google.com/github/alilouati/data-visualization-using-python/blob/main/Chapter_2/Ch2_Plotting_Libraries.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

    # Chapter 2: Plotting Libraries: Matplotlib, Seaborn, Plotly & Altair
    **Data Visualization — Ph.D. in Computing** · Dr. Ali Louati · IS Dept., PSAU

    ⏱️ Estimated lab time: **2 × 90 min**

    ### Learning objectives
    By the end of this lab you should be able to:
    1. Use Matplotlib's object-oriented API (Figure/Axes) instead of the implicit `pyplot` state machine.
2. Reproduce and correct the classic business charts (stacked/grouped bars, lines, areas, histograms).
3. Use Seaborn's statistical plots (regression with CI, joint/pair plots, box/violin, hexbin, 2-D KDE) and small multiples.
4. Build interactive figures with Plotly Express and declarative, linked views with Altair (grammar of graphics).
5. Produce reproducible, publication-quality vector figures sized for a journal column.
6. Choose the right library for a task and justify the choice.

    ### How to use this notebook
    * Run cells top to bottom with **Shift + Enter** (or *Runtime → Run all*).
    * Cells marked **✏️ Exercise** are for you. Cells marked **🔑 Solution** are collapsed: try first, then double-click to reveal.
    * Save your own copy: *File → Save a copy in Drive*.

## 0. Setup

In [ ]:
import sys, numpy as np, pandas as pd, matplotlib as mpl, matplotlib.pyplot as plt, seaborn as sns
import plotly.express as px, altair as alt
from matplotlib.ticker import FuncFormatter
IN_COLAB = "google.colab" in sys.modules
print("plotly", px.__version__ if hasattr(px, "__version__") else "", "| altair", alt.__version__, "| seaborn", sns.__version__)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(7)

## 1. Matplotlib: the two APIs
`plt.plot(...)` (implicit, *pyplot*) is convenient but relies on hidden global state. For anything beyond a quick look, use the **object-oriented** API: create a `Figure` and one or more `Axes`, and call methods on them. It scales to multi-panel figures and functions that draw into a given `ax`.

### 1.1 Anatomy of a figure

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
x = np.linspace(0, 10, 200)
ax.plot(x, np.sin(x), label="sin(x)")
ax.plot(x, np.cos(x), label="cos(x)", linestyle="--")
ax.set(title="Axes title", xlabel="x label (units)", ylabel="y label (units)")
ax.legend(title="legend"); ax.grid(True, alpha=.3)
ax.annotate("local max", xy=(np.pi/2, 1), xytext=(3, 1.2), arrowprops=dict(arrowstyle="->"))
ax.set_ylim(-1.3, 1.5)
fig.suptitle("Figure suptitle"); fig.tight_layout(); plt.show()

### 1.2 Stacked and grouped bars (corrected version of the slide code)
The slide version had curly quotes (`’`), `plt.grid(true)` (Python needs `True`) and an undefined `bar_width`. Here is a clean OO version. Note that *stacking* revenue and earnings is conceptually wrong (earnings are part of revenue) — the grouped version is the honest one.

In [ ]:
year = np.array([2011, 2012, 2013, 2014, 2015])
revenue = np.array([20, 25, 30, 35, 40]); earning = np.array([10, 15, 20, 25, 30])
billions = FuncFormatter(lambda v, pos: f"${v:.0f}b")
x, w = np.arange(len(year)), 0.38

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
a1.bar(x, revenue, 0.6, label="Revenue"); a1.bar(x, earning, 0.6, bottom=revenue, label="Earning")
a1.set_title("Stacked (misleading here: earnings ⊂ revenue)")
a2.bar(x - w/2, revenue, w, label="Revenue"); a2.bar(x + w/2, earning, w, label="Earning")
a2.set_title("Grouped (honest comparison)")
for a in (a1, a2):
    a.set_xticks(x, year); a.yaxis.set_major_formatter(billions); a.legend(frameon=False)
a1.set_ylabel("Billions of US $"); fig.suptitle("MSFT — annual data (illustrative values)")
fig.tight_layout(); plt.show()

### 1.3 Lines, stacked areas and why pies are rarely the best choice

In [ ]:
months = pd.period_range("2024-01", periods=12, freq="M").to_timestamp()
season = 300 + 220 * np.sin((np.arange(12) - 3) / 12 * 2 * np.pi)   # summer peak (A/C load)
usage = pd.DataFrame({c: season * s + rng.normal(0, 15, 12) for c, s in [("Riyadh", 1.0), ("Jeddah", 0.8), ("Alkhobar", 0.55)]}, index=months)

fig, axes = plt.subplots(1, 3, figsize=(16, 3.6))
usage.plot(ax=axes[0], marker="o"); axes[0].set_title("Line: compare trends"); axes[0].set_ylabel("GWh (synthetic)")
axes[1].stackplot(usage.index, usage.T.values, labels=usage.columns, alpha=.85); axes[1].legend(loc="upper left")
axes[1].set_title("Stacked area: total + composition")
tot = usage.sum().sort_values()
axes[2].barh(tot.index, tot.values); axes[2].set_title("Annual total: sorted bar beats a pie")
for a in axes[:2]: a.tick_params(axis="x", rotation=45)
fig.tight_layout(); plt.show()

> **Critique.** In a stacked area only the bottom series has a common baseline; the others are hard to read. When the individual series matter more than the total, prefer **small multiples**.

### 1.4 Histograms

In [ ]:
applications = rng.normal(loc=60, scale=12, size=1000)  # e.g. applicants per job post
fig, ax = plt.subplots(figsize=(6, 3.4))
ax.hist(applications, bins="fd", edgecolor="white")
ax.axvline(np.median(applications), color="C3", label=f"median = {np.median(applications):.0f}")
ax.set(xlabel="applications per post", ylabel="count", title="Histogram (Freedman–Diaconis bins)"); ax.legend()
plt.show()

## 2. Seaborn: statistical graphics on top of Matplotlib
Seaborn is **data-frame aware** and **long-format first**: you map columns to `x`, `y`, `hue`, `col`, `row`. *Figure-level* functions (`relplot`, `displot`, `catplot`, `lmplot`) create whole grids of small multiples; *axes-level* functions (`scatterplot`, `histplot`, …) draw into an existing `ax`.

### 2.1 Distributions on Iris

In [ ]:
iris = sns.load_dataset("iris")
long = iris.melt(id_vars="species", var_name="measurement", value_name="cm")
sns.displot(long, x="cm", hue="species", col="measurement", kind="kde", fill=True,
            common_norm=False, facet_kws=dict(sharex=False, sharey=False), height=2.6, col_wrap=4)
plt.show()

In [ ]:
sns.pairplot(iris, hue="species", corner=True, height=1.8, plot_kws=dict(s=15, alpha=.7))
plt.show()

### 2.2 Relationship + model: Titanic
Survival is **binary**, so a linear regression line (as in the original slides) is the wrong model. Compare it with a **logistic** fit, both with bootstrap confidence bands.

In [ ]:
titanic = sns.load_dataset("titanic").dropna(subset=["age", "fare"])
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
sns.scatterplot(data=titanic, x="age", y="fare", hue="survived", alpha=.6, ax=axes[0])
axes[0].set_yscale("log"); axes[0].set_title("Age vs fare (log scale)")
sns.regplot(data=titanic, x="age", y="survived", logistic=True, y_jitter=.03,
            scatter_kws=dict(alpha=.15, s=10), ax=axes[1], n_boot=200, seed=0)
axes[1].set_title("P(survived | age) — logistic fit, 95 % CI")
fig.tight_layout(); plt.show()

In [ ]:
# Small multiples: does the age effect differ by sex and class?
sns.lmplot(data=titanic, x="age", y="survived", col="class", hue="sex", logistic=True,
           height=2.8, scatter=False, n_boot=100, seed=0)
plt.show()

### 2.3 Tips: joint, hexbin and 2-D density views

In [ ]:
tips = sns.load_dataset("tips")
g = sns.jointplot(data=tips, x="total_bill", y="tip", kind="reg", height=4, marginal_kws=dict(bins=20))
g.figure.suptitle("Joint plot: scatter + regression + marginals", y=1.02); plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
hb = axes[0].hexbin(tips.total_bill, tips.tip, gridsize=18, cmap="viridis", mincnt=1)
fig.colorbar(hb, ax=axes[0], label="count"); axes[0].set(title="Hexbin", xlabel="total bill", ylabel="tip")
sns.kdeplot(data=tips, x="total_bill", y="tip", fill=True, cmap="mako", thresh=.05, ax=axes[1])
sns.scatterplot(data=tips, x="total_bill", y="tip", s=8, color="w", ax=axes[1]); axes[1].set_title("2-D KDE + points")
fig.tight_layout(); plt.show()

In [ ]:
# Better question: what tip *rate* do people leave?
tips["tip_pct"] = 100 * tips.tip / tips.total_bill
sns.relplot(data=tips, x="total_bill", y="tip_pct", col="time", hue="smoker", height=3, alpha=.7)
plt.show()

### 2.4 Categorical distributions: box → violin → raw data
A box plot hides multimodality and sample size. Show the data.

In [ ]:
diamonds = sns.load_dataset("diamonds")
order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
fig, axes = plt.subplots(1, 3, figsize=(16, 3.6), sharey=True)
sns.boxplot(data=diamonds, x="cut", y="price", order=order, ax=axes[0], fliersize=1); axes[0].set_title("Box")
sns.violinplot(data=diamonds, x="cut", y="price", order=order, ax=axes[1], inner="quart", cut=0); axes[1].set_title("Violin")
sns.boxenplot(data=diamonds, x="cut", y="price", order=order, ax=axes[2]); axes[2].set_title("Letter-value (boxen) for big n")
for a in axes: a.tick_params(axis="x", rotation=20)
fig.tight_layout(); plt.show()

> **Simpson-style trap.** "Ideal" diamonds have a *lower* median price than "Premium". Why? Because cut is confounded with **carat**. Check it:

In [ ]:
g = sns.relplot(data=diamonds.sample(8000, random_state=0), x="carat", y="price", hue="cut", hue_order=order,
                palette="viridis", s=6, alpha=.5, height=3.6, aspect=1.4)
g.set(xscale="log", yscale="log")
for a in g.axes.flat:
    a.xaxis.set_minor_formatter(mpl.ticker.NullFormatter()); a.yaxis.set_minor_formatter(mpl.ticker.NullFormatter())
g.figure.suptitle("Price vs carat (log–log), coloured by cut", y=1.03); plt.show()
print(diamonds.groupby("cut", observed=True)["carat"].median().reindex(order))

## 3. Plotly Express: interactivity for free
Hover, zoom, pan, legend toggling and export are built in. Plotly figures render natively in Colab.

In [ ]:
gap = px.data.gapminder()
fig = px.scatter(gap, x="gdpPercap", y="lifeExp", size="pop", color="continent", hover_name="country",
                 animation_frame="year", animation_group="country", log_x=True, size_max=45,
                 range_x=[200, 1.2e5], range_y=[20, 90], height=480,
                 title="Gapminder — press ▶ to animate")
fig.show()

In [ ]:
fig = px.histogram(tips, x="tip_pct", color="time", marginal="box", nbins=40, barmode="overlay", opacity=.7,
                   title="Tip % by time (interactive histogram + marginal box)")
fig.show()

## 4. Altair: a declarative grammar of graphics with linked views
In Altair you declare **data → mark → encodings** (Wilkinson's *Grammar of Graphics*, Vega-Lite). Interaction is declared too: below, a brush on the scatter plot filters the bar chart (**brushing & linking**). Drag on the left chart.

In [ ]:
cars = sns.load_dataset("mpg").dropna()
brush = alt.selection_interval(encodings=["x", "y"])
points = alt.Chart(cars).mark_point().encode(
    x="horsepower:Q", y="mpg:Q",
    color=alt.condition(brush, "origin:N", alt.value("lightgray")),
    tooltip=["name", "model_year"]).add_params(brush).properties(width=330, height=260)
bars = alt.Chart(cars).mark_bar().encode(
    x="count():Q", y="origin:N", color="origin:N").transform_filter(brush).properties(width=330, height=120)
points | bars

## 5. Publication-quality, reproducible figures
Journals ask for **vector** formats, embedded fonts and exact widths (e.g. IEEE single column ≈ 3.5 in, double ≈ 7.16 in). Put the style in one place and export to PDF/SVG.

In [ ]:
PUB = {"figure.figsize": (3.5, 2.4), "font.size": 8, "axes.titlesize": 9, "axes.labelsize": 8,
       "legend.fontsize": 7, "xtick.labelsize": 7, "ytick.labelsize": 7, "lines.linewidth": 1.2,
       "pdf.fonttype": 42, "ps.fonttype": 42, "savefig.bbox": "tight", "savefig.dpi": 300}
with mpl.rc_context(PUB), sns.axes_style("ticks"):
    fig, ax = plt.subplots()
    sns.lineplot(data=sns.load_dataset("fmri"), x="timepoint", y="signal", hue="region", errorbar=("ci", 95),
                 n_boot=200, seed=0, ax=ax)
    ax.set(xlabel="time (s)", ylabel="BOLD signal", title="Mean ± 95 % CI"); sns.despine()
    fig.savefig("fig_fmri.pdf"); fig.savefig("fig_fmri.svg"); fig.savefig("fig_fmri.png")
    plt.show()
import os; print({f: f"{os.path.getsize(f)/1024:.0f} KB" for f in ["fig_fmri.pdf", "fig_fmri.svg", "fig_fmri.png"]})

In [ ]:
# In Colab, download the PDF to your computer:
if IN_COLAB:
    from google.colab import files
    files.download("fig_fmri.pdf")

## 6. Which library when?
| Need | Matplotlib | Seaborn | Plotly | Altair |
|---|---|---|---|---|
| Total control, print/PDF | ★★★ | ★★ | ★ | ★★ |
| Statistical summaries (CI, KDE, regression) | ★ | ★★★ | ★★ | ★★ |
| Interactivity / web | ★ | ★ | ★★★ | ★★★ |
| Linked views declared in code | – | – | ★ (via Dash) | ★★★ |
| Big data (> 1 M points) | ★★ (rasterize) | ★ | ★★ (WebGL) | ★ (needs aggregation) |
| Feeds dashboards (Ch. 4–5) | ★ | ★ | ★★★ | ★★ |

## 7. ✏️ Exercises
**E1 — Same figure, four libraries.** Using `penguins`, plot bill length vs bill depth coloured by species, with a per-species regression line, in Matplotlib, Seaborn, Plotly and Altair. Report lines of code and what each made easy/hard.

**E2 — Simpson's paradox.** Show that the *overall* bill length–depth correlation in `penguins` is negative while the within-species correlations are positive. One figure must make this obvious to a reader in 5 seconds.

**E3 — Reproduce a figure from a paper in your field** as a 3.5-inch-wide vector PDF. Critique the original.

In [ ]:
# ✏️ E2 — your code here
penguins = sns.load_dataset("penguins").dropna()

In [ ]:
#@title 🔑 Solution E2 (try first, then double-click to reveal) { display-mode: "form" }
print("overall r =", round(penguins.bill_length_mm.corr(penguins.bill_depth_mm), 2))
print(penguins.groupby("species").apply(lambda d: round(d.bill_length_mm.corr(d.bill_depth_mm), 2), include_groups=False))
fig, ax = plt.subplots(figsize=(6, 4))
sns.regplot(data=penguins, x="bill_length_mm", y="bill_depth_mm", scatter=False, color="k", ax=ax,
            line_kws=dict(ls="--"), label="all penguins", n_boot=100, seed=0)
for sp, d in penguins.groupby("species"):
    sns.regplot(data=d, x="bill_length_mm", y="bill_depth_mm", ax=ax, label=sp, scatter_kws=dict(s=12), n_boot=100, seed=0)
ax.legend(); ax.set_title("Simpson's paradox: the pooled trend reverses within species"); plt.show()

## References
* Hunter, J. D. (2007). *Matplotlib: A 2D graphics environment*. CiSE 9(3).
* Waskom, M. (2021). *seaborn: statistical data visualization*. JOSS 6(60).
* VanderPlas, J. et al. (2018). *Altair: Interactive Statistical Visualizations for Python*. JOSS 3(32).
* Satyanarayan, A. et al. (2017). *Vega-Lite: A Grammar of Interactive Graphics*. IEEE TVCG.
* Wilkinson, L. (2005). *The Grammar of Graphics*, 2nd ed. Springer.
* Rougier, N. (2021). *Scientific Visualization: Python + Matplotlib* (open access).